In [ ]:
# Install required vision-language libraries and optimize CUDA memory 

import subprocess, os
subprocess.run(["pip", "install", "-q", "qwen-vl-utils", "transformers==4.49.0",
                "peft", "accelerate", "bitsandbytes", "datasets",
                "pillow", "rouge-score", "nltk"], check=True)

print("All required packages are installed now")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch
torch.cuda.empty_cache()


qwen-vl-utils - Utilities for Qwen Vision-Language models

transformers==4.49.0 - HuggingFace Transformers (specific version for compatibility)

peft - Parameter-Efficient Fine-Tuning for LoRA

accelerate - Multi-GPU / mixed precision training helper

bitsandbytes - 4-bit / 8-bit quantization (reduces VRAM usage)
datasets - HuggingFace dataset loading

pillow - Image processing

rouge-score	- Evaluation metric (text generation quality)

nltk - NLP utilities (tokenization, etc.)

max_split_size_mb:128 - Prevent PyTorch from splitting GPU memory into very tiny chunks.

In [ ]:

# test is 100 samples and full is 5700 samples

RUN_MODE = "full"   

if RUN_MODE == "test":
    TRAIN_SAMPLES = 100
    VAL_SAMPLES   = 10
    NUM_EPOCHS    = 3
    LOG_EVERY     = 5
    print("testing the pipeline")
else:
    TRAIN_SAMPLES = 6000
    VAL_SAMPLES   = 500
    NUM_EPOCHS    = 7        
    LOG_EVERY     = 100
    print("full training mode")

SAVE_DIR = "/kaggle/working/qwen_vqa_v5"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:

# Imports & Config

import re, json, ast, random, time
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image

from datasets import load_dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Dataset as TorchDataset, DataLoader

import nltk
nltk.download("punkt", quiet=True)

SEED          = 42
DATASET_NAME  = "sujet-ai/Sujet-Finance-QA-Vision-100k"
MODEL_NAME    = "Qwen/Qwen2-VL-2B-Instruct"
CONTENT_LIMIT = 800   # used at both train & inference 
MAX_SEQ_LEN   = 1024  # token limit passed to the tokeniser
BATCH_SIZE    = 1
ACCUM_STEPS   = 8     # gradient accumulation
LR            = 5e-6  # lowered from 1e-4 and then to 2e-5
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
DEVICE        = "cuda"
DTYPE         = torch.float16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


AdamW - commonly used in Transformers because it applies weight decay correctly 

OneCycleLR - learning rate scheduler that increases the learning rate first and then decreases it during training 

TorchDataset, DataLoader - define how your data is structured and to load it in batches during training. 

In [ ]:
# Load the qwen model in float16
# AutoProcessor handles both the image pre-processor and the text tokeniser.

print("\nLoading the model")
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    use_fast=False, # use slow python based tokenizer/processor 
    min_pixels=128*28*28,
    max_pixels=256*28*28,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16, # 2B params × 2 bytes = 4GB, float32 will use 8GB
    device_map="auto",
    low_cpu_mem_usage=True, # loads model weights in chunks to CPU then GPU 
)
print("Model is loaded now")

# LoRA rank changed from r =8 to r=16 and also alpha doubled to 32 
# added k_proj + o_proj to improve performance

lora_config = LoraConfig(
    r=16,
    lora_alpha=32, # how strong the lora influnces the model
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



In [ ]:
# Load the Dataset

# parse_qa_pairs() creates a list of dictionarie of all the qa pairs tagged to every image

def parse_qa_pairs(qa_str):
    if not qa_str: return []
    if isinstance(qa_str, list): return qa_str
    try: pairs = json.loads(qa_str)
    except Exception:
        try: pairs = ast.literal_eval(qa_str)
        except Exception: return []
    if isinstance(pairs, dict): pairs = [pairs]
    if not isinstance(pairs, list): return []
    results = []
    for p in pairs:
        if not isinstance(p, dict): continue
        q = p.get("question") or p.get("ques") or p.get("q")
        a = p.get("answer")   or p.get("ans")  or p.get("a")
        if q and a:
            results.append({"question": str(q).strip(), "answer": str(a).strip()})
    return results

# Remove markdown formatting as the dataset stores OCR content with markdown formatting

def clean_content(content):
    if not content: return ""
    content = re.sub(r"#{1,3}\s*", "", content)
    content = re.sub(r"\*\*(.+?)\*\*", r"\1", content)
    content = re.sub(r"\*(.+?)\*", r"\1", content)
    content = re.sub(r"^\s*[-•]\s*", "", content, flags=re.MULTILINE)
    content = re.sub(r"\n+", " ", content)
    content = re.sub(r"\s+", " ", content)
    return content.strip()

print()
print("Loading the dataset")
dataset = load_dataset(DATASET_NAME)

flat_images, flat_questions, flat_answers, flat_contents = [], [], [], []
for ex in dataset["train"]:
    content  = clean_content(ex.get("content", ""))
    qa_pairs = parse_qa_pairs(ex.get("qa_pairs"))
    for qa in qa_pairs:
        flat_images.append(ex["image"])
        flat_questions.append(qa["question"])
        flat_answers.append(qa["answer"])
        flat_contents.append(content)
        if len(flat_images) >= 6000: # limiting to load only 6000qa pairs instead of 100,000+ qa pairs
            break
    if len(flat_images) >= 6000:
        break

print(f"Total QA pairs loaded: {len(flat_images):,}")

# Shuffling the data
indices = list(range(len(flat_images)))
random.shuffle(indices)

# Creating train and test split
n_train = min(TRAIN_SAMPLES, int(len(indices) * 0.9))
n_val   = min(VAL_SAMPLES,   len(indices) - n_train)
tr_idx  = indices[:n_train]
va_idx  = indices[n_train:n_train + n_val]

print(f"train: {len(tr_idx):,} and val: {len(va_idx):,}")

In [ ]:

# Dataset & DataLoader
# Training using IMAGE + OCR content 

class FinanceVQADataset(TorchDataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        img = flat_images[i]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        return {
            "image":    img.convert("RGB"),
            "question": flat_questions[i],
            "answer":   flat_answers[i],
            "content":  flat_contents[i],
        }

def build_prompt_text(item, include_content=True):

    if include_content and item["content"]:
        text = (
            "You are an expert in proccessing financial document.\n"
            f"Document content: {item['content'][:CONTENT_LIMIT]}\n\n"
            "Answer the question using only information from the document. "
            "Give a direct and complete answer.\n\n"
            f"Question: {item['question']}"
        )
    else:
        text = (
            "You are an expert in proccessing financial document."
            "Answer the question based on the document image.\n\n"
            f"Question: {item['question']}"
        )
    return text

def collate_fn(batch):
    from qwen_vl_utils import process_vision_info

    all_input_ids, all_attention_mask = [], []
    all_labels, all_pixel_values, all_image_grid_thw = [], [], []

    for item in batch:
        
        full_messages = [
            {"role": "user", "content": [
                {"type": "image", "image": item["image"]},
                {"type": "text",  "text": build_prompt_text(item, include_content=True)},
            ]},
            {"role": "assistant", "content": item["answer"]},
        ]
        prompt_messages = full_messages[:-1]

        full_text   = processor.apply_chat_template(full_messages,   tokenize=False, add_generation_prompt=False)
        prompt_text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        # Prepares the image in the special format Qwen-VL expects
        image_inputs, _ = process_vision_info(full_messages)

        enc = processor(
            text=[full_text], images=image_inputs,
            return_tensors="pt", padding=False,
            truncation=True, max_length=MAX_SEQ_LEN,  # chnaged to 1024 from 512
        )

        input_ids  = enc["input_ids"][0]
        labels     = input_ids.clone()
        labels[:]  = -100  

        # unmask answer tokens only
        prompt_enc = processor.tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)
        prompt_len = min(len(prompt_enc["input_ids"][0]), len(labels))
        labels[prompt_len:] = input_ids[prompt_len:]

        all_input_ids.append(input_ids)
        all_attention_mask.append(enc["attention_mask"][0])
        all_labels.append(labels)
        all_pixel_values.append(enc["pixel_values"])
        all_image_grid_thw.append(enc["image_grid_thw"])

    max_len = max(x.shape[0] for x in all_input_ids) # x.shape[0] - gives th len
    pad_id  = processor.tokenizer.pad_token_id or 0
    B = len(batch)

    batch_input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    batch_attn_mask = torch.zeros((B, max_len),          dtype=torch.long)
    batch_labels    = torch.full((B, max_len), -100,     dtype=torch.long)

    for i, (ids, mask, lbls) in enumerate(zip(all_input_ids, all_attention_mask, all_labels)):
        L = ids.shape[0]
        batch_input_ids[i, :L] = ids
        batch_attn_mask[i, :L] = mask
        batch_labels[i, :L]    = lbls

    return {
        "input_ids":      batch_input_ids.to(DEVICE),
        "attention_mask": batch_attn_mask.to(DEVICE),
        "labels":         batch_labels.to(DEVICE),
        "pixel_values":   torch.cat(all_pixel_values, dim=0).to(DEVICE, dtype=DTYPE),
        "image_grid_thw": torch.cat(all_image_grid_thw, dim=0).to(DEVICE),
    }

train_ds     = FinanceVQADataset(tr_idx)
val_ds       = FinanceVQADataset(va_idx)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)
print(f"train batches: {len(train_loader)} and val batches: {len(val_loader)}")


Qwen’s forward method expects: (this is what collate() returns) <br>

model( <br>
    input_ids=..., <br>
    attention_mask=..., <br>
    pixel_values=..., <br>
    image_grid_thw=..., <br>
    labels=... <br>
)

In [ ]:

# Sanity check 

print("running sanity check")

with torch.no_grad():
    out = model(**test_batch)
print(f"sanity check passed and loss is: {out.loss.item():.4f}")

del test_batch
torch.cuda.empty_cache()

In [ ]:

# Optimizer & Scheduler

optimizer   = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = max(1, (len(train_loader) * NUM_EPOCHS) // ACCUM_STEPS)
# pct_start chnaged from 0.2 to 0.05 
scheduler   = OneCycleLR(optimizer, max_lr=LR, total_steps=total_steps, pct_start=0.05)
scaler      = torch.amp.GradScaler('cuda', enabled=True, growth_interval=200)
print(f"Total steps: {total_steps} | Effective batch: {BATCH_SIZE * ACCUM_STEPS}")

total steps = 5400 * 7 // 8 = 4725
rise for 236 optimizer updates, then decay for 4489 updates

In [ ]:
# Training Loop

def evaluate_loss(model, loader, max_batches=20):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= max_batches: break
            try:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    out = model(**batch)
                if out.loss is not None:
                    total += out.loss.item(); n += 1
            except Exception: pass
    return total / n if n else float("inf")

def save_checkpoint(model, processor, epoch, val_loss):
    path = os.path.join(SAVE_DIR, f"epoch_{epoch}_valloss_{val_loss:.3f}")
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    processor.save_pretrained(path)
    print(f"moded has been saved → {path}")
    return path

def run_inference(model, processor, sample):
    
    from qwen_vl_utils import process_vision_info
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": sample["image"]},
        {"type": "text",  "text": build_prompt_text(sample, include_content=True)},
    ]}]
    prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(msgs)
    inputs = processor(text=[prompt], images=image_inputs, return_tensors="pt", padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype=DTYPE)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False, repetition_penalty=1.1)
    pred = processor.tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    return pred

train_losses, val_losses = [], []
best_val, best_ckpt, global_step = float("inf"), None, 0


print("Qwen2-VL training")

for epoch in range(NUM_EPOCHS):
    model.train()
    ep_loss, t0 = 0.0, time.time()
    optimizer.zero_grad()

    for bi, batch in enumerate(train_loader):
        try:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = model(**batch).loss / ACCUM_STEPS

            if torch.isnan(loss) or torch.isinf(loss):
                optimizer.zero_grad()
                continue

            scaler.scale(loss).backward()

            if (bi + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            ep_loss += loss.item() * ACCUM_STEPS

            if (bi + 1) % LOG_EVERY == 0:
                avg = ep_loss / (bi + 1)
                cur = loss.item() * ACCUM_STEPS
                print(f"[Ep{epoch+1}/{NUM_EPOCHS}] loss {cur:.4f}")
                train_losses.append(avg)

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                optimizer.zero_grad()
            else:
                raise

    avg_train = ep_loss / max(1, len(train_loader))
    avg_val   = evaluate_loss(model, val_loader)
    val_losses.append(avg_val)
    print()
    print(f"Epoch {epoch+1} | Train: {avg_train:.4f} and Val: {avg_val:.4f}")

    if avg_val < best_val:
        best_val  = avg_val
        best_ckpt = save_checkpoint(model, processor, epoch+1, avg_val)
        print(f"best val loss: {best_val:.4f}")

    latest_path = os.path.join(SAVE_DIR, "latest_checkpoint")
    model.save_pretrained(latest_path)
    processor.save_pretrained(latest_path)
    print(f"Latest checkpoint has been updated")

    # Per-epoch sample prediction
    model.eval()
    sample = val_ds[0]
    pred = run_inference(model, processor, sample)
    print(f"[Ep{epoch+1}] Q   : {sample['question']}")
    print(f"[Ep{epoch+1}] Ref : {sample['answer']}")
    print(f"[Ep{epoch+1}] Pred: {pred}\n")
    model.train()

print()
print(f"Training has been completed! Best val loss: {best_val:.4f} is at {best_ckpt}")



In [ ]:

# Loss Curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
if train_losses:
    ax1.plot(train_losses, "b-"); ax1.set_title("Train Loss"); ax1.grid(True, alpha=0.3)
if val_losses:
    ax2.plot(range(1, len(val_losses)+1), val_losses, "r-o"); ax2.set_title("Val Loss"); ax2.grid(True, alpha=0.3)
plt.suptitle(f"Qwen2-VL Training with Image + OCR ")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "loss_curves.png"), dpi=130)
plt.show()